In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('/content/gurgaon_properties_post_feature_selection.csv')
df.head()

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category,price
0,0,36,3.0,2.0,2,1,850.0,0.0,0.0,0.0,1,1,0.82
1,0,95,2.0,2.0,2,1,1226.0,1.0,0.0,0.0,1,2,0.95
2,0,103,2.0,2.0,1,1,1000.0,0.0,0.0,0.0,1,0,0.32
3,0,99,3.0,4.0,4,3,1615.0,1.0,0.0,1.0,0,2,1.60
4,0,5,2.0,2.0,1,3,582.0,0.0,1.0,0.0,0,2,0.48


In [4]:
# We will apply linear reg as baseline model , but for applying linear reg. we have to do one hot encoding of categorical cols
# as oridnal encoding will not give good results with linear reg.

In [5]:
# 1. one hot encoding
# 2. scaling
# 3. log transformation of price

In [6]:
# one hot encode -> sector, balcony, agePossession, furnishing type, Luxury category, floor category

In [7]:
X = df.drop(columns=['price'])
y = df['price']

In [8]:
from sklearn.model_selection import KFold , cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.compose import ColumnTransformer

In [18]:
columns_to_encode = ['sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [19]:
# applying log transformation to target variable
y_transformed = np.log1p(y)

In [20]:
# creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num' , StandardScaler() , ['property_type', 'bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat' , OneHotEncoder(drop='first') , columns_to_encode)
    ],
    remainder='passthrough'
)

In [21]:
# creating a pipeline
pipeline = Pipeline([
    ('preprocessor' , preprocessor),
    ('regressor' , LinearRegression())
])

In [22]:
# K-Fold cross validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline , X , y_transformed , cv = kfold , scoring='r2')

In [23]:
scores.mean()

np.float64(0.8558123475287653)

In [24]:
scores.std()

np.float64(0.015558381492447235)

In [25]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

In [26]:
pipeline.fit(X_train , y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['property_type', 'bedRoom',
                                                   'bathroom', 'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['sector', 'balcony',
                                                   'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category'])])),
                ('regressor', LinearRegression())])

In [27]:
y_pred = pipeline.predict(X_test)

In [28]:
y_pred = np.expm1(y_pred)

In [29]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.expm1(y_test) , y_pred)

0.6483879307359236

In [31]:
# on an average our model does a mistake of 64 lac , which is high, we have work on it.
# lest try with thoda advance algo with basic default params.

In [32]:
from sklearn.svm import SVR

In [33]:
# creating a pipeline
pipeline1 = Pipeline([
    ('preprocessor' , preprocessor),
    ('regressor' , SVR())
])

In [34]:
# K-Fold cross validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline1 , X , y_transformed , cv = kfold , scoring='r2')

In [35]:
scores.mean()

np.float64(0.8845360715052788)

In [36]:
scores.std()

np.float64(0.014784881452419994)

In [37]:
pipeline1.fit(X_train , y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['property_type', 'bedRoom',
                                                   'bathroom', 'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['sector', 'balcony',
                                                   'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category'])])),
                ('regressor', SVR())])

In [41]:
y_pred1 = pipeline1.predict(X_test)

In [42]:
y_pred1 = np.expm1(y_pred1)

In [43]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.expm1(y_test) , y_pred1)

0.5324591082613235

In [44]:
# we are able improve r2 from 85 to 88 and reduce MAE from 64 to 53 lac with just different algo and no hyperparameter tuning.

In [45]:
# What can be done to improve performance
# 1. Different algorithms can be used.
# 2. Hyperparameter tuning.
# 3. Feature engineering.
# 4. Bring more data.